In [2]:
# RUN
# importing
from bs4 import BeautifulSoup
import requests
import asyncio
import os
import uuid
from dotenv import load_dotenv
import psycopg
import aiohttp

# pinecone
from pinecone import Pinecone, ServerlessSpec


# langchain
from langchain_openai.chat_models import ChatOpenAI  # model
from langchain_core.output_parsers import StrOutputParser  # parser
from langchain.prompts import ChatPromptTemplate  # prompt
from langchain.text_splitter import RecursiveCharacterTextSplitter  # text splitter
from langchain_core.documents import Document  # document class
from langchain_pinecone import PineconeVectorStore  # vector store class
from langchain_openai import OpenAIEmbeddings  # embedder

In [3]:
# api keys
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
conn_string = os.environ.get("DB_CONNECTION_STRING")

# args for vectorDB object
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("bettercallpaul-descriptions-index")
documents_index = pc.Index("bettercallpaul-documents-index")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


# creating vectorDB object
vectorDB = PineconeVectorStore(
    index=index,
    embedding=embeddings,
)

documentsVectorDB = PineconeVectorStore(
  index=documents_index,
  embedding=embeddings
)

# creating langchain_components
model = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-3.5-turbo")
parser = StrOutputParser()
template = """
          You are a legal assistant specialized in Canadian federal legislation.

          You will be given:
          1. A user question about Canadian Acts or Regulations.
          2. A set of retrieved document excerpts (“context”).
          3. Document NAMES AND LINKS for the reference section.

          Your strict instructions:
          - You MUST answer ONLY using information found in the provided CONTEXT.
          - If the context does not contain enough information to answer confidently, say:
            "I cannot answer based on the retrieved documents."
          - Do NOT invent legal definitions, rules, penalties, commentary, or interpretations
            that are not explicitly stated in the context.
          - If the user asks for legal advice, warnings, or interpretations beyond the text, say:
            "I can only provide information found in the retrieved documents."
          - Keep the answer concise (3–6 sentences), neutral, and purely informational.

          ### REQUIRED OUTPUT FORMAT
          You MUST produce BOTH sections below.  
          Do not omit, rename, or reorder them.

          Example Output Format:
          ANSWER:
          <your 3–6 sentence answer here>

          REFERENCE:
          Documents' Names: <documents' names>
          Links: <documents' links>

          ### Now answer using ONLY the information below.

          ---------------------
          CONTEXT:
          {context}
          ---------------------

          QUESTION:
          {question}

          NAME:
          {name}

          LINK:
          {link}

          ### Write your output now following the REQUIRED OUTPUT FORMAT strictly.
          ANSWER:
          """

prompt = ChatPromptTemplate.from_template(template)

chain = prompt | model | parser

In [4]:
def existing_titles(conn_string):
  """
  returns list of titles found within 
  """
  with psycopg.connect(conn_string) as conn:
    with conn.cursor() as cur:
      cur.execute("SELECT doc_titles FROM document_information;")
      records = cur.fetchall()
      existing_titles_list = [record[0] for record in records]

  return existing_titles_list

In [5]:
def filter_retrieval(query):
  """
  Retrieves 5 documents similar to query and returns valid documents based on similarity scores.

  Return:
  valid_documents: list of document names and links formatted as dictionaries
  """

  # retrieving documents to filter through
  results = vectorDB.similarity_search_with_relevance_scores(
      f"{query}",
      k = 5
  )

  # valid documents to scrape and input in documents index
  valid_documents = []

  for i in range(len(results)):
     print(results[i][1])
     if results[i][1] >= 0.70:
        valid_documents.append({
           "name": results[i][0].metadata["doc_title"],
           "link": results[i][0].metadata["doc_link"]
        })

  return valid_documents

In [6]:
def compare(valid_documents, existing_titles_list, conn_string):
  """
  returns valid_documents who's namespace is currently not in database
  """

  new_additions = []

  for valid_document in valid_documents:
    if valid_document["name"] not in existing_titles_list:
      new_additions.append(valid_document)
  
  adding_to_db = []

  for addition in new_additions:
    adding_to_db.append((addition["name"],))

  with psycopg.connect(conn_string) as conn:
    with conn.cursor() as cur:
      insert_query = """
                  INSERT INTO document_information (doc_titles)
                  VALUES (%s);
              """

      cur.executemany(insert_query, adding_to_db)

      conn.commit()
      
  return new_additions 

In [7]:
SEMAPHORE = asyncio.Semaphore(10)

async def create_soup(name, link, batch_soups, session):
  try:
    async with SEMAPHORE:
            async with session.get(f"{link[:-11]}/FullText.html") as resp:
                scrapable_text = await resp.text()
    scrapable_soup = BeautifulSoup(scrapable_text, "html.parser")

    batch_soups.append((name, link, scrapable_soup))
  except Exception as e:
    print(f"Error on {name}: {e}")

In [8]:
async def create_scrapables(new_additions):
  async with aiohttp.ClientSession() as session:
    batch_soups = []

    # tasks to be executed in current batch
    batch_tasks = []

    for i in range(len(new_additions)):
      name = new_additions[i]["name"]
      link = new_additions[i]["link"]

      batch_tasks.append(asyncio.create_task(create_soup(name, link, batch_soups, session)))

    await asyncio.gather(*batch_tasks)

    return batch_soups

In [9]:
def scrape_scrapables(batch_soups):
  BATCH_CHUNKS = []

  # for each soup
  for i in range(len(batch_soups)):
    parse_list = []

    name = batch_soups[i][0]
    link = batch_soups[i][1]
    scrapable_soup = batch_soups[i][2]


    # parse the soup
    for item in scrapable_soup.find_all(["p", "h2", "h3", "h4"]):
          if (item.name == "p"):
            parse_list.extend([s for s in item.stripped_strings if s != "Marginal note:"])
          
          elif item.name == "h2" and "Part" in item.get("class", []):
                  parse_list.extend([s for s in item.stripped_strings if s != "Marginal note:"])

          elif item.name in ("h3", "h4") and "Subheading" in item.get("class", []):
              parse_list.extend([s for s in item.stripped_strings if s != "Marginal note:"])
    
    
    parsed_string = " ".join(parse_list)
    
    # create parsed string into Document object with proper metadata
    current_doc = Document(page_content=parsed_string, metadata={
      "doc_title":name,
      "doc_link":link,
    })

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1400, chunk_overlap=200)

    # chunk parsed sting Document
    current_chunks = text_splitter.split_documents([current_doc])

    # insert into BATCH_CHUNKS
    BATCH_CHUNKS.extend(current_chunks)

  return BATCH_CHUNKS

In [10]:
def embedd_chunks(BATCH_CHUNKS):
  embedding_size = 500

  # creating embeddings for entire batch of 50, by sending batches of 2048 chunks to openAI embedder
  embedded_chunks = []
  for j in range(0, len(BATCH_CHUNKS), embedding_size):
      embedding_batch_chunks = BATCH_CHUNKS[j:j+embedding_size]
      texts = [chunk.page_content for chunk in embedding_batch_chunks]

      embedding_batch_embedds = embeddings.embed_documents(texts)
      embedded_chunks.extend(embedding_batch_embedds)
  
  return embedded_chunks

In [11]:
def upsert_chunks(BATCH_CHUNKS, embedded_chunks, documents_index):
  UPSERT_BATCH = 100

  for j in range(0, len(embedded_chunks), UPSERT_BATCH):
      upsert_batch_chunks = BATCH_CHUNKS[j:j+UPSERT_BATCH]
      upsert_batch_embedds = embedded_chunks[j:j+UPSERT_BATCH]

      vectors = []
      for doc, embed in zip(upsert_batch_chunks, upsert_batch_embedds):
          vectors.append({
              "id": str(uuid.uuid4()),
              "values": embed,
              "metadata": {
              "text": doc.page_content,        
              **doc.metadata                   
          }
          })

      documents_index.upsert(vectors=vectors)

In [12]:
def retrieve_context(filtering_names, query, documentsVectorDB):
  relevant_chunks = documentsVectorDB.similarity_search(
    query,
    k=10, 
    filter=filtering_names
  )

  return relevant_chunks

In [13]:
def generate_answer(chain, query, relevant_chunks, valid_documents):
  generated_answer = chain.invoke({
    "question": f"{query}", 
    "context": "\n\n".join([chunk.page_content for chunk in relevant_chunks]),
    "name": [valid_document["name"] for valid_document in valid_documents], 
    "link": [valid_document["link"] for valid_document in valid_documents]  
  })

  return generated_answer

In [14]:
async def answer(query, conn_string, documents_index, documentsVectorDB):
  """
  Retrieves current namespaces present in vector database.
  Retrieves documents similar to query provided.
  """
  
  valid_documents = filter_retrieval(query)

  # if no valid documents, return with lack of confidence
  if len(valid_documents) == 0:
    return "I’m not confident which law your question relates to."
  
  # if valid documents, identify documents non-existent in documents_index
  existing_titles_list = existing_titles(conn_string)

  # valid_documents to add to vdb
  new_additions = compare(valid_documents, existing_titles_list, conn_string)

  # new document chunks to be inserted into documentsVectorDB
  if len(new_additions) > 0:
    # creating batch_soups
    batch_soups = await create_scrapables(new_additions)

    print(len(batch_soups))

    # scraping batch_soups into chunks
    BATCH_CHUNKS = scrape_scrapables(batch_soups)

    print(f"length of chunks created to be inserted: {len(BATCH_CHUNKS)}")

    # embedding chunks
    embedded_chunks = embedd_chunks(BATCH_CHUNKS)

    # upserting chunks
    upsert_chunks(BATCH_CHUNKS, embedded_chunks, documents_index)
  
  # returning answer

  filtering_names =  {
    "doc_title": {
      "$in": [valid_document["name"] for valid_document in valid_documents]}
    }
  
  # retrieving chunks for context
  relevant_chunks = retrieve_context(filtering_names, query, documentsVectorDB)

  # generating answer
  generated_answer = generate_answer(chain, query, relevant_chunks, valid_documents)
  
  return generated_answer

  

In [ ]:
query = "Tell me about Canadian child labour laws?"

print("Paul is thinking...")
generated_answer = await answer(query, conn_string, documents_index, documentsVectorDB)
print(generated_answer)

Paul is thinking...
0.771856278
0.762963295
0.7618260385
0.7424669415
0.735171795
